In [ ]:

from functions.load_data_function import generate_dataset,load_data,generate_T_train_test_dataset,fig_plot
import torch
from functions.config import Config
from functions.model import CNN_Transformer_kan, CNN_Transformer,CNN_Baseline,CNN_LSTM_Baseline
import os
from functions.train_function_4 import model_train_without_T
from functions.train_function_5 import model_train_with_S
from functions.train_function_6 import model_train_without_S

import numpy as np
import torch.optim as optim
from openTSNE import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, IterableDataset

In [ ]:
final_data_path='../T_Toyota_S10_T1'
#final_data_path='../T_HNEL_S_IECON_Toyota_XJTU_data_S3_T1'
#final_data_path='../T_HNEL_S10_T1'
#final_data_path='../T_TongJi_S_XJTU_data_S1_T1'
S_data_dict=load_data(os.path.join(final_data_path,'S_data_dict.pkl'))
S_soh_dict=load_data(os.path.join(final_data_path,'S_soh_dict.pkl'))
T_data=load_data(os.path.join(final_data_path,'T_data.pkl'))
T_data_soh=load_data(os.path.join(final_data_path,'T_soh.pkl'))
T_train_data=load_data(os.path.join(final_data_path,'T_train_data.pkl'))
T_test_data=load_data(os.path.join(final_data_path,'T_test_data.pkl'))
T_train_soh=load_data(os.path.join(final_data_path,'T_train_soh.pkl'))
T_test_soh=load_data(os.path.join(final_data_path,'T_test_soh.pkl'))

config=Config()

#print(S_data_dict.keys())
#print(T_data.shape)
#print(S_data_dict['HNEL'].shape)

#dict_S_loader, T_train_loader, T_test_loader = generate_T_train_test_dataset(S_data_dict, S_soh_dict, T_data, T_data_soh, config)
dict_S_loader,T_loader,indexed_T_train_loader,T_train_loader,T_test_loader,indexed_T_test_loader = generate_dataset(S_data_dict, S_soh_dict, T_data,T_train_data,T_test_data, T_data_soh,T_train_soh,T_test_soh, config)

In [ ]:
#target_dataset = 'HNEL'  # 选择目标数据集
#target_dataset = 'SNL_NMC'  # 选择目标数据集
#target_dataset = 'Oxford'  # 选择目标数据集
target_dataset = 'NASA'  # 选择目标数据集

T_cell_cycles=load_data(os.path.join(final_data_path,'T_cell_cycles.pkl'))
dict_S_cell_cycles=load_data(os.path.join(final_data_path,'dict_S_cell_cycles.pkl'))
T_train_cell_cycles=load_data(os.path.join(final_data_path,'T_train_cell_cycles.pkl'))
T_test_cell_cycles=load_data(os.path.join(final_data_path,'T_test_cell_cycles.pkl'))
#print(dict_S_cell_cycles['XJTU_battery'][0])


In [ ]:
for S_domain, loader in dict_S_loader.items():
    print(f"Loader for domain {S_domain}: Number of batches = {len(loader)}")

print(f"Loader for domain {target_dataset}: Number of batches = {len(T_loader)}")


In [ ]:
model = CNN_Transformer_kan(config).to(config.Device)
#model=CNN_Baseline(config).to(config.Device)
#model=CNN_Transformer(config).to(config.Device)
#model=CNN_LSTM_Baseline(config).to(config.Device)
optimizer = optim.Adam(model.parameters(), config.Learning_rate)  # 优化器设置


scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1, end_factor=0.1, total_iters=350),  # 前期快速收敛
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=0.01, total_iters=100),  # 后期缓慢衰减
        #torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=350, eta_min=1e-5), # 后期精细调整
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=0.001, total_iters=350)  # 后期缓慢衰减

    ],
    milestones=[350,450]    # 前300 epoch用StepLR
)
scheduler=torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1, end_factor=0.01, total_iters=config.N_epoch)

In [ ]:
file_name='target_CALCE_version_2'


In [ ]:
#S_loader=dict_S_loader['IECON']
#S_data=S_data_dict['IECON']
#print(S_data.shape) 
#print(len(S_loader))

In [ ]:
#model_train_with_S(config, dict_S_loader, indexed_T_train_loader,T_train_loader,T_test_loader, model,dict_S_cell_cycles,T_test_cell_cycles,T_train_cell_cycles,S_data_dict,S_soh_dict, optimizer,scheduler)

In [ ]:

model = CNN_Transformer_kan(config).to(config.Device)
optimizer = optim.Adam(model.parameters(), config.Learning_rate)  # 优化器设置
#scheduler=torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1, end_factor=0.08, total_iters=config.N_epoch)

model_train_without_S(config,dict_S_loader,indexed_T_train_loader,T_train_loader,T_test_loader,model,T_test_cell_cycles,T_train_cell_cycles,optimizer,scheduler)
